## 1、RunnableSequence
作用：构造一个串行的执行链，通过RunnableSequence的实例，调用invoke方法，就等于链当中每一个组件去调用invoke，然后将调用结果传递给下一个组件

In [17]:
from langchain_core.runnables import RunnableSequence
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
import dotenv
dotenv.load_dotenv()
from langchain_core.output_parsers import StrOutputParser
prompt_template = PromptTemplate(input_variables=["user_info"],template="{user_info}")
openai = ChatOpenAI()
output_parser = StrOutputParser()
# chain = prompt_template | openai | output_parser
# 1、实例化
runnable_sequence = RunnableSequence(prompt_template,openai,output_parser)

In [18]:
runnable_sequence.invoke("你好，你是谁")

'我是一款人工智能助手，可以回答你的问题和提供帮助。有什么可以帮助你的吗？'

In [5]:
chain = prompt_template | openai | output_parser
type(chain)

langchain_core.runnables.base.RunnableSequence

## 2、RunnableParallel
作用：对于同一个输入，能够并行去执行多个组件。

In [7]:
from langchain_core.runnables import RunnableParallel
def func1(a1):
    return a1+"__func1_output"
def func2(a1):
    return a1+"__func2_output"
runnable_parallel = RunnableParallel({"key1":func1,"key2":func2})

In [8]:
runnable_parallel.invoke("你好")

{'key1': '你好__func1_output', 'key2': '你好__func2_output'}

### 具体应用：对于用户输入的同一个问题，我们想要调用不同的大模型进行回答，用户可以比对不同大模型回答的效果

In [9]:
from langchain_openai import ChatOpenAI
from langchain_deepseek import ChatDeepSeek
from langchain_core.prompts import ChatPromptTemplate
# 两个模型的实例
openai_model = ChatOpenAI(
    model="gpt-4o-mini"
)
deepseek = ChatDeepSeek(model="deepseek-chat")
messages = [
    ("system","你是一个数学家"),
    ("user","{user_question}")
]
# template的实例
message_template = ChatPromptTemplate.from_messages(messages)
# 结合LCEL和RunnableParallel来去构造一个简单的应用，实现 两个模型之间效果的对比
chain = message_template | RunnableParallel({"openai_output":openai_model,"deepseek_output":deepseek})
chain

ChatPromptTemplate(input_variables=['user_question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='你是一个数学家'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['user_question'], input_types={}, partial_variables={}, template='{user_question}'), additional_kwargs={})])
| {
    openai_output: ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000020C25316FC0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000020C253164E0>, root_client=<openai.OpenAI object at 0x0000020C24126BA0>, root_async_client=<openai.AsyncOpenAI object at 0x0000020C25314530>, model_name='gpt-4o-mini', model_kwargs={}, openai_api_key=SecretStr('**********')),
    deepseek_output: ChatDeepSeek(client=<openai.resources.chat.completions.completions.Completions object at 0x0000020C2

In [10]:
chain.invoke({"user_question":"什么是哥德巴赫猜想"})

{'openai_output': AIMessage(content='哥德巴赫猜想是数论中的一个著名未解问题，由德国数学家克里斯蒂安·哥德巴赫在1742年首次提出。它主要有两个版本：\n\n1. **强哥德巴赫猜想**：每个大于2的偶数都可以表示为两个素数之和。也就是说，对于任何偶数 \\( n > 2 \\)，存在素数 \\( p \\) 和 \\( q \\)，使得 \\( n = p + q \\)。\n\n2. **弱哥德巴赫猜想**（又称为奇数哥德巴赫猜想）：每个大于5的奇数都可以表示为三个素数之和。即对于任何奇数 \\( n > 5 \\)，存在素数 \\( p \\)，\\( q \\) 和 \\( r \\)，使得 \\( n = p + q + r \\)。\n\n尽管哥德巴赫猜想在数学界已有多个数值验证和部分成果，但至今仍未被完全证明或反驳。强哥德巴赫猜想对偶数的表述经过了大量的计算验证，甚至计算机已验证到非常大的范围，但仍需要一个完整的数学证明。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 261, 'prompt_tokens': 24, 'total_tokens': 285, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b', 'id': 'chatcmpl-CZ7Fa1O0mD9UzPZnhaFhukBGUIldd', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run

## RunnableLambda
作用：能够传入一个Python的函数，或者说可执行的一个对象，将其封装成Runnable的实例，使得可以通过invoke来进行调用

In [11]:
from langchain_core.runnables import RunnableLambda
runnable_lambda = RunnableLambda(func1)

In [13]:
runnable_lambda = RunnableLambda(lambda x: x+"__lambda_x")

In [14]:
runnable_lambda.invoke("你好")

'你好__lambda_x'

## RunnablePassthrough
1. 作用：可以透传，输入什么，就输出什么
2. 可以为输入，添加额外的键，通过调用RunnablePassthrough.assign()方法指定新的键的执行逻辑

In [19]:
from langchain_core.runnables import RunnablePassthrough
runnable_passthrough = RunnablePassthrough()

In [20]:
runnable_passthrough.invoke({"key":"value"})

{'key': 'value'}

In [26]:
from langchain_core.runnables import RunnablePassthrough
## 通过assign，可以为传递给runnable passthrough的变量，输出添加额外的键

## 注意，此处能够将第一个dict写到chain当中，就是因为Runnable底层，实现了__ror__方法
chain = {
    "text1": lambda x: x + " world",
    "text2": lambda x: x + ", how are you",
} | RunnablePassthrough.assign(word_c5454545ount=lambda x : len(x["text1"]))

result = chain.invoke("hello")
result

{'text1': 'hello world',
 'text2': 'hello, how are you',
 'word_c5454545ount': 11}

## RunnableBranch
1. 作用：其实就是一个if_else操作,能够根据我们所写的判断条件，去具体执行某一个分支的逻辑
2. 构造方式，传入多个分支组成的列表，列表的每一个元素都是一个元组，元组的第一个元素，是一个判断条件，元组的第二个元素，是命中当前判断条件时，所执行的逻辑
3. 当调用invoke方法时，会自上向下去判断，是否符合条件，如果符合条件，就进行执行，如果不符合，会落到最终的“else”分支去执行

In [29]:
from langchain_core.runnables import RunnableBranch
branch = RunnableBranch(
    (lambda x: isinstance(x, str), lambda x: x.upper()),# if x是字符串类型，就执行 x.upper()
    (lambda x: isinstance(x, int), lambda x: x + 1), # elif x是int类型，就执行 x+1 操作
    (lambda x: isinstance(x, float), lambda x: x * 2), # elif x 是float类型，就执行 x*2
    lambda x: "goodbye", # else: 前面的条件都没有命中，就走else 这部分的逻辑
)

In [33]:
branch.invoke(ChatOpenAI)

'goodbye'

## RunnableWithFallbacks
作用：在前面的runnable出现异常时，就会执行runnableWithFallbacks

In [38]:
from langchain_openai import ChatOpenAI
import dotenv
dotenv.load_dotenv()
llm = ChatOpenAI()
chain = PromptTemplate.from_template("hello") | llm
# 会报错
chain.invoke("hello")

TypeError: Expected mapping type as input to PromptTemplate. Received <class 'str'>.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT 

In [45]:
# 调用方式：通过调用runnable组件的with_fallbacks方法，就可以得到一个RunnableWithFallbacks的一个实例
# with_fallbacks能够接收一个列表，在chain前面报错时，会依次，逐个执行链条当中的runnable组件，如果第一个报错，就执行第二个，以此类推。直到最后一个，如果一直报错，那么就抛出最开始的错误
def func1(x):
    raise Exception(x)
def func2(x):
    raise Exception(x)
chain_with_fallback = chain.with_fallbacks([RunnableLambda(func1)])
type(chain_with_fallback)
chain_with_fallback.invoke("hello")

TypeError: Expected mapping type as input to PromptTemplate. Received <class 'str'>.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT 

In [ ]:
# try:
#     func1("sd")
# except FileNotFoundError as e:
#     # 去执行某些操作
#     pass
# except ValueError as e:
#     # 去执行某些操作
#     pass
# else:
#     raise Exception(e)